**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Gas Challenge Statistics — Subject-wise Comparison

Loads per-subject maps from `InVivo_SmoothFirst_v1.ipynb` and plots ROI-mean
parameter values per condition (air / hyper / hypo) for three methods:

- **Box + strip** plot with individual subject paired lines
- **Wilcoxon signed-rank** (or paired t-test) with FDR correction

**Layout** (set `PLOT_STYLE`): `'by_method'` | `'by_param'` | `'combined'`

In [ ]:
import os, glob, json, warnings, re
import numpy as np
import pandas as pd
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from itertools import combinations

warnings.filterwarnings('ignore', category=RuntimeWarning)

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DL_4P  = '#1565C0'
C_TRIPLE = '#6A1B9A'
C_DM     = '#E65100'
METHOD_COLORS = {'DL-4p': C_DL_4P, 'Triple (A+B+C)': C_TRIPLE, 'DM': C_DM}

COND_ORDER  = ['air', 'hyper', 'hypo']
COND_LABELS = {'air': 'Normoxia\n(Air)', 'hyper': 'Hyperoxia', 'hypo': 'Hypoxia'}
COND_COLORS = {'air': '#37474F', 'hyper': '#EF6C00', 'hypo': '#1976D2'}

print('Imports OK')

## 2. Configuration — **edit paths here**

In [ ]:
CONFIG = {
    # ── Input ─────────────────────────────────────────────────────────────
    # Option A: load from the pre-built CSV (fast)
    'csv_path'    : './results/invivo_smooth_v1/roi_statistics_smooth.csv',

    # Option B: re-extract from .mat files if CSV is missing / stale
    'maps_dir'    : './results/invivo_smooth_v1',
    'maps_pattern': '*_maps_smooth.mat',

    # ── Output ────────────────────────────────────────────────────────────
    'output_dir'  : './results/invivo_smooth_v1/gas_stats',

    # ── Plot style: 'by_method' | 'by_param' | 'combined' ────────────────
    'plot_style'  : 'by_method',

    # ── Statistics ────────────────────────────────────────────────────────
    # 'wilcoxon' (non-parametric, recommended for small N)
    # 'ttest'    (paired t-test)
    'stat_test'   : 'wilcoxon',
    'alpha'       : 0.05,
    'fdr_correct' : True,    # Benjamini-Hochberg FDR across all pairs x params

    # ── ROI extraction (used when re-reading .mat files) ──────────────────
    'param_names' : ['SO2', 'CBV', 'R', 'T2'],
    'param_scale' : [100,   100,   1e6,  1000],
    'param_units' : ['%',   '%',   'um', 'ms'],
    'method_keys' : ['DL_4p', 'Triple', 'DM'],
    'method_names': ['DL-4p', 'Triple (A+B+C)', 'DM'],

    # ── Figure options ────────────────────────────────────────────────────
    'show_subject_lines' : True,
    'show_strip'         : True,
    'box_width'          : 0.45,
    'strip_alpha'        : 0.85,
    'line_alpha'         : 0.35,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
FIG_DIR = CONFIG['output_dir']
methods = CONFIG['method_names']
params  = CONFIG['param_names']
print(f'Output: {FIG_DIR}')

## 3. Load / extract ROI data

In [ ]:
def load_mat_safe(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat: return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def key_to_subject_id(img_key):
    m = re.match(r'(?:img_)?(e\d+)', img_key, flags=re.IGNORECASE)
    return m.group(1).upper() if m else img_key

def key_to_condition(img_key):
    raw = img_key.lower()
    for tag, label in [('hyper','hyper'),('hypo','hypo'),('air','air'),('norm','air')]:
        if tag in raw: return label
    return 'unknown'

def extract_roi_from_mats(maps_dir, pattern, config):
    rows = []
    files = sorted(glob.glob(os.path.join(maps_dir, pattern)))
    print(f'Found {len(files)} .mat files')
    for fpath in files:
        basename  = os.path.splitext(os.path.basename(fpath))[0]
        img_key   = re.sub(r'_maps_smooth$', '', basename)
        subj_id   = key_to_subject_id(img_key)
        condition = key_to_condition(img_key)
        print(f'  {img_key}  subj={subj_id}  cond={condition}')
        try:
            mask = load_mat_safe(fpath, 'mask').astype(bool)
        except Exception:
            print(f'    WARNING: no mask in {fpath}, skipping'); continue
        flat_mask = mask.flatten()
        for mkey, mname in zip(config['method_keys'], config['method_names']):
            try:
                arr = load_mat_safe(fpath, mkey).reshape(-1, 4)
            except Exception:
                print(f'    WARNING: {mkey} not found'); continue
            for pi, (pname, sc, punit) in enumerate(
                    zip(config['param_names'], config['param_scale'], config['param_units'])):
                vals = arr[flat_mask, pi] * sc
                vals = vals[np.isfinite(vals)]
                if not len(vals): continue
                rows.append({
                    'subject'  : subj_id,
                    'condition': condition,
                    'method'   : mname,
                    'parameter': pname,
                    'unit'     : punit,
                    'mean'     : float(np.nanmean(vals)),
                    'std'      : float(np.nanstd(vals)),
                    'median'   : float(np.nanmedian(vals)),
                    'n_voxels' : len(vals),
                })
    return pd.DataFrame(rows)

csv_path = CONFIG['csv_path']
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'Loaded CSV: {csv_path}  ({len(df)} rows)')
    if 'median' not in df.columns:
        df['median'] = df['mean']
else:
    print('CSV not found — extracting from .mat files...')
    df = extract_roi_from_mats(CONFIG['maps_dir'], CONFIG['maps_pattern'], CONFIG)
    df.to_csv(csv_path, index=False)
    print(f'CSV saved to {csv_path}')

conditions = [c for c in COND_ORDER if c in df['condition'].unique()]
print(f'Subjects  : {sorted(df["subject"].unique())}')
print(f'Conditions: {conditions}')
print(f'Methods   : {sorted(df["method"].unique())}')
print(f'Parameters: {sorted(df["parameter"].unique())}')
df.head(8)

## 4. Statistical tests

In [ ]:
def run_pairwise_tests(df, method, parameter, conditions, stat_test='wilcoxon'):
    results = {}
    sub_df = df[(df['method'] == method) & (df['parameter'] == parameter)]
    cond_subjs = {c: set(sub_df[sub_df['condition'] == c]['subject']) for c in conditions}
    common_subjs = sorted(set.intersection(*cond_subjs.values()))
    if len(common_subjs) < 3:
        print(f'  WARNING: only {len(common_subjs)} paired subjects for {method}/{parameter}')
    for c1, c2 in combinations(conditions, 2):
        v1, v2 = [], []
        for s in common_subjs:
            r1 = sub_df[(sub_df['condition']==c1) & (sub_df['subject']==s)]['mean']
            r2 = sub_df[(sub_df['condition']==c2) & (sub_df['subject']==s)]['mean']
            if len(r1) and len(r2):
                v1.append(r1.values[0]); v2.append(r2.values[0])
        n = min(len(v1), len(v2))
        v1, v2 = np.array(v1[:n]), np.array(v2[:n])
        if n < 3:
            results[(c1,c2)] = {'p_raw': np.nan, 'statistic': np.nan, 'n': n}; continue
        try:
            if stat_test == 'wilcoxon':
                stat, p = stats.wilcoxon(v1, v2, alternative='two-sided')
            else:
                stat, p = stats.ttest_rel(v1, v2)
            results[(c1,c2)] = {'p_raw': float(p), 'statistic': float(stat), 'n': n}
        except Exception:
            results[(c1,c2)] = {'p_raw': np.nan, 'statistic': np.nan, 'n': n}
    return results

def fdr_correct(p_values):
    valid = [(i, p) for i, p in enumerate(p_values) if np.isfinite(p)]
    if not valid: return p_values
    m = len(valid)
    sorted_v = sorted(valid, key=lambda x: x[1])
    adj = np.array(p_values, dtype=float)
    for rank, (i, p) in enumerate(sorted_v, 1):
        adj[i] = min(p * m / rank, 1.0)
    for rank in range(len(sorted_v)-2, -1, -1):
        adj[sorted_v[rank][0]] = min(adj[sorted_v[rank][0]], adj[sorted_v[rank+1][0]])
    return adj.tolist()

def p_to_stars(p):
    if np.isnan(p) or p >= 0.05: return 'ns'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    return '*'

# ── Run all tests ─────────────────────────────────────────────────────────
STATS = {}
all_keys, all_praw = [], []
for method in methods:
    STATS[method] = {}
    for param in params:
        res = run_pairwise_tests(df, method, param, conditions,
                                 stat_test=CONFIG['stat_test'])
        STATS[method][param] = res
        for pair, r in res.items():
            all_keys.append((method, param, pair))
            all_praw.append(r['p_raw'])

adj = fdr_correct(all_praw) if CONFIG['fdr_correct'] else all_praw
for (method, param, pair), p_adj in zip(all_keys, adj):
    STATS[method][param][pair]['p_adj'] = p_adj
    STATS[method][param][pair]['stars'] = p_to_stars(p_adj)

print(f'Test: {CONFIG["stat_test"]}  |  FDR: {CONFIG["fdr_correct"]}')
print(f'{"Method":<22} {"Param":<6} {"Pair":<16} {"p_raw":>8} {"p_adj":>8} {"sig":>5}')
print('-'*72)
for method in methods:
    for param in params:
        for pair, r in STATS[method][param].items():
            p_r = f"{r['p_raw']:.4f}" if np.isfinite(r.get('p_raw', np.nan)) else 'nan'
            p_a = f"{r['p_adj']:.4f}" if np.isfinite(r.get('p_adj', np.nan)) else 'nan'
            print(f'{method:<22} {param:<6} {pair[0]+" vs "+pair[1]:<16} {p_r:>8} {p_a:>8} {r["stars"]:>5}')

## 5. Plot helpers

In [ ]:
def get_param_label(param):
    units  = dict(zip(CONFIG['param_names'], CONFIG['param_units']))
    labels = {'SO2': 'SO₂', 'CBV': 'CBV', 'R': 'R', 'T2': 'T2'}
    return f"{labels.get(param, param)} ({units.get(param, '')})"

def draw_bracket(ax, x1, x2, y, text, color='#333333', h_frac=0.015, fs=8):
    if text == 'ns': return
    yspan = ax.get_ylim()[1] - ax.get_ylim()[0]
    h = yspan * h_frac
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.0, color=color, clip_on=False)
    ax.text((x1+x2)/2, y+h*1.1, text, ha='center', va='bottom',
            fontsize=fs, color=color, fontweight='bold')

def plot_panel(ax, df, method, param, conditions, stats_dict, config):
    sub = df[(df['method']==method) & (df['parameter']==param)].copy()
    x_pos = {c: i for i, c in enumerate(conditions)}
    n_cond = len(conditions)
    data = {}
    for c in conditions:
        rows = sub[sub['condition']==c].sort_values('subject')
        data[c] = {'subjects': rows['subject'].tolist(), 'means': rows['mean'].tolist()}

    # Box plot
    bp = ax.boxplot(
        [data[c]['means'] for c in conditions],
        positions=[x_pos[c] for c in conditions],
        widths=config['box_width'],
        patch_artist=True,
        medianprops=dict(color='#0D47A1', linewidth=2.5, solid_capstyle='round'),
        whiskerprops=dict(color='#666', linewidth=1.2),
        capprops=dict(color='#666', linewidth=1.2),
        flierprops=dict(marker=''),
        boxprops=dict(linewidth=1.0),
    )
    for patch, c in zip(bp['boxes'], conditions):
        patch.set_facecolor(COND_COLORS[c]); patch.set_alpha(0.5)

    # Paired lines
    if config['show_subject_lines']:
        all_s = sorted(set(s for c in conditions for s in data[c]['subjects']))
        for subj in all_s:
            xs, ys = [], []
            for c in conditions:
                if subj in data[c]['subjects']:
                    idx = data[c]['subjects'].index(subj)
                    xs.append(x_pos[c]); ys.append(data[c]['means'][idx])
            if len(xs) > 1:
                ax.plot(xs, ys, '-', color='#888', lw=0.9,
                        alpha=config['line_alpha'], zorder=2)

    # Strip
    if config['show_strip']:
        rng = np.random.default_rng(42)
        for c in conditions:
            vals = data[c]['means']
            jit  = rng.uniform(-0.06, 0.06, len(vals))
            ax.scatter([x_pos[c]+j for j in jit], vals,
                       s=35, color=COND_COLORS[c], edgecolors='white',
                       linewidths=0.5, zorder=5, alpha=config['strip_alpha'])

    # ── Global mean +/- SEM trend line across conditions ─────────────────
    group_means = []
    group_sems  = []
    for c in conditions:
        vals = data[c]['means']
        if vals:
            group_means.append(np.mean(vals))
            group_sems.append(np.std(vals, ddof=1) / np.sqrt(len(vals)))
        else:
            group_means.append(np.nan)
            group_sems.append(np.nan)

    gx = [x_pos[c] for c in conditions]
    # ax.plot(gx, group_means, '-', color='#111111', lw=2.0,
    #         zorder=7, solid_capstyle='round')
    # ax.plot(gx, group_means, 'D', color='#111111', markersize=6,
    #         zorder=8, markeredgecolor='white', markeredgewidth=0.8)
    # # Error bars (SEM)
    # ax.errorbar(gx, group_means, yerr=group_sems,
    #             fmt='none', ecolor='#111111', elinewidth=1.2,
    #             capsize=3, capthick=1.2, zorder=8)
    ax.plot(gx, group_means, '-', color='#111111', lw=2.0,
            zorder=7, solid_capstyle='round')
    ax.plot(gx, group_means, 'D', color='#111111', markersize=6,
            zorder=8, markeredgecolor='white', markeredgewidth=0.8)

    # ── Y limits and significance brackets
    all_vals = [v for c in conditions for v in data[c]['means']]
    if all_vals:
        ymax = np.nanmax(all_vals); ymin = np.nanmin(all_vals)
        yspan = max(ymax - ymin, 1.0)
        ax.set_ylim(ymin - 0.08*yspan, ymax + 0.40*yspan)

    pairs_sorted = sorted(combinations(range(n_cond), 2))
    yrange_plot  = ax.get_ylim()[1] - np.nanmax(all_vals)
    step = yrange_plot / (len(pairs_sorted) + 1)
    for lvl, (i, j) in enumerate(pairs_sorted):
        pair_key = (conditions[i], conditions[j])
        stars = stats_dict.get(pair_key, {}).get('stars', 'ns')
        y_brk = np.nanmax(all_vals) + step*(lvl + 0.7)
        draw_bracket(ax, x_pos[conditions[i]], x_pos[conditions[j]],
                     y_brk, stars)

    ax.set_xticks([x_pos[c] for c in conditions])
    ax.set_xticklabels([COND_LABELS.get(c, c) for c in conditions], fontsize=8.5)
    ax.set_ylabel(get_param_label(param), fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

def save_fig(fig, name):
    for ext in ['png', 'pdf']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext=='png' else None)
    print(f'  Saved: {name}')

print('Plot helpers ready.')

## 6. Figures — one per method (1 row x 4 params)

In [ ]:
method_colors = [C_DL_4P, C_TRIPLE, C_DM]
handles_legend = [mpatches.Patch(color=COND_COLORS[c], alpha=0.7,
                                  label=COND_LABELS[c].replace('\n',' '))
                  for c in conditions]

for method, mcolor in zip(methods, method_colors):
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.8),
                             gridspec_kw={'wspace': 0.44})
    fig.suptitle(method, fontsize=12, fontweight='bold',
                 color=mcolor, y=1.03)
    for ax, param in zip(axes, params):
        plot_panel(ax, df, method, param, conditions,
                   STATS[method][param], CONFIG)
        ax.set_title(get_param_label(param), fontsize=10, fontweight='bold')
        ax.set_ylabel('')
    axes[0].set_ylabel('Mean ROI value', fontsize=9)
    fig.legend(handles=handles_legend, loc='upper right',
               bbox_to_anchor=(1.01, 1.0), fontsize=8, frameon=False)
    plt.tight_layout()
    tag = method.replace(' ','_').replace('(','').replace(')','')
    save_fig(fig, f'gas_bymethod_{tag}')
    plt.show()

## 7. Figures — one per parameter (1 row x 3 methods)

In [ ]:
for param in params:
    fig, axes = plt.subplots(1, len(methods), figsize=(11, 3.8),
                             gridspec_kw={'wspace': 0.44})
    fig.suptitle(get_param_label(param), fontsize=12, fontweight='bold', y=1.03)
    for ax, method, mcolor in zip(axes, methods, method_colors):
        plot_panel(ax, df, method, param, conditions,
                   STATS[method][param], CONFIG)
        ax.set_title(method, fontsize=10, fontweight='bold', color=mcolor)
        if ax is not axes[0]: ax.set_ylabel('')
    fig.legend(handles=handles_legend, loc='upper right',
               bbox_to_anchor=(1.01, 1.0), fontsize=8, frameon=False)
    plt.tight_layout()
    save_fig(fig, f'gas_byparam_{param}')
    plt.show()

## 8. Combined figure (3 methods x 4 params)

In [ ]:
fig, axes = plt.subplots(
    len(methods), len(params),
    figsize=(14, 10),
    gridspec_kw={'wspace': 0.38, 'hspace': 0.58}
)
for ri, (method, mcolor) in enumerate(zip(methods, method_colors)):
    for ci, param in enumerate(params):
        ax = axes[ri, ci]
        plot_panel(ax, df, method, param, conditions,
                   STATS[method][param], CONFIG)
        if ri == 0:
            ax.set_title(get_param_label(param), fontsize=10, fontweight='bold')
        if ci == 0:
            ax.set_ylabel(method, fontsize=9, fontweight='bold', color=mcolor)
        else:
            ax.set_ylabel('')

fig.legend(handles=handles_legend, loc='upper right',
           bbox_to_anchor=(1.01, 0.98), fontsize=9, frameon=False)
save_fig(fig, 'gas_combined')
plt.show()

In [ ]:
import os, glob

GM_MASKS_DIR = '../GESFIDE_data/GES_ROI'
print("All files in masks dir:")
for f in sorted(os.listdir(GM_MASKS_DIR)):
    print(f)

In [ ]:
def find_mask(subj_id):
    masks_dir = '../GESFIDE_data/GES_ROI'   # ← hardcode directly here
    for sfx in ['', '_AIR', '_HYPER', '_HYPO', '_air']:
        for ext in ['.nii.gz', '.nii']:
            p = os.path.join(masks_dir, f'{subj_id}{sfx}_ROI{ext}')
            if os.path.exists(p): return p
    raise FileNotFoundError(f'No mask for {subj_id} in {masks_dir}')

In [ ]:
# ── Gray Matter — Gas Challenge (reuses plot_panel helpers) ───────────────
import nibabel as nib

GM_MASKS_DIR = '../GESFIDE_data/GES_ROI'   # ← adjust if needed

def find_gm_mask(subj_id):
    for sfx in ['', '_AIR', '_HYPER', '_HYPO', '_air']:
        for ext in ['.nii.gz', '.nii']:
            p = os.path.join(GM_MASKS_DIR, f'{subj_id}{sfx}_ROI{ext}')
            if os.path.exists(p): return p
    raise FileNotFoundError(f'No GM mask for {subj_id} in {GM_MASKS_DIR}')

# ── Extract ───────────────────────────────────────────────────────────────
rows_gm = []
for fpath in sorted(glob.glob(os.path.join(CONFIG['maps_dir'], CONFIG['maps_pattern']))):
    basename  = os.path.splitext(os.path.basename(fpath))[0]
    img_key   = re.sub(r'_maps_smooth$', '', basename)
    subj_id   = key_to_subject_id(img_key)
    condition = key_to_condition(img_key)
    if condition not in COND_ORDER: continue
    try:
        gm_nii  = nib.load(find_gm_mask(subj_id)).get_fdata()
        # gm_flat = ((gm_nii[..., 0] if gm_nii.ndim == 4 else gm_nii) > 0).flatten()
        seg = (gm_nii[..., 0] if gm_nii.ndim == 4 else gm_nii)
        gm_flat = (seg == 1).flatten()   # label 2 = Gray Matter
    
    except FileNotFoundError as e:
        print(f'  SKIP: {e}'); continue
    for mkey, mname in zip(CONFIG['method_keys'], CONFIG['method_names']):
        try:
            arr = load_mat_safe(fpath, mkey).reshape(-1, 4)
        except Exception: continue
        for pi, (pname, sc, punit) in enumerate(
                zip(CONFIG['param_names'], CONFIG['param_scale'], CONFIG['param_units'])):
            vals = arr[gm_flat, pi] * sc
            vals = vals[np.isfinite(vals)]
            if not len(vals): continue
            rows_gm.append({'subject': subj_id, 'condition': condition,
                            'method': mname, 'parameter': pname, 'unit': punit,
                            'mean': float(np.nanmean(vals)),
                            'std':  float(np.nanstd(vals)),
                            'n_voxels': len(vals)})

df_gm = pd.DataFrame(rows_gm)
df_gm.to_csv(os.path.join(FIG_DIR, 'roi_gm_allconds.csv'), index=False)
print(f'GM rows: {len(df_gm)}  subjects: {sorted(df_gm["subject"].unique())}')

# ── Stats ─────────────────────────────────────────────────────────────────
conditions_gm = [c for c in COND_ORDER if c in df_gm['condition'].unique()]
STATS_GM = {}
all_keys_gm, all_praw_gm = [], []
for method in methods:
    STATS_GM[method] = {}
    for param in params:
        res = run_pairwise_tests(df_gm, method, param, conditions_gm,
                                 stat_test=CONFIG['stat_test'])
        STATS_GM[method][param] = res
        for pair, r in res.items():
            all_keys_gm.append((method, param, pair))
            all_praw_gm.append(r['p_raw'])

adj_gm = fdr_correct(all_praw_gm) if CONFIG['fdr_correct'] else all_praw_gm
for (method, param, pair), p_adj in zip(all_keys_gm, adj_gm):
    STATS_GM[method][param][pair]['p_adj'] = p_adj
    STATS_GM[method][param][pair]['stars'] = p_to_stars(p_adj)

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    len(methods), len(params),
    figsize=(14, 3.6 * len(methods)),
    gridspec_kw={'wspace': 0.38, 'hspace': 0.55}
)
for ri, (method, mcolor) in enumerate(zip(methods, method_colors)):
    for ci, param in enumerate(params):
        ax = axes[ri, ci]
        plot_panel(ax, df_gm, method, param, conditions_gm,
                   STATS_GM[method][param], CONFIG)
        if ri == 0:
            ax.set_title(get_param_label(param), fontsize=10, fontweight='bold')
        if ci == 0:
            ax.set_ylabel(method, fontsize=9, fontweight='bold', color=mcolor)
        else:
            ax.set_ylabel('')

fig.suptitle('Gray Matter — Gas Challenge', fontsize=12, fontweight='bold', y=1.01)
fig.legend(handles=handles_legend, loc='upper right',
           bbox_to_anchor=(1.01, 0.98), fontsize=9, frameon=False)
plt.tight_layout()
save_fig(fig, 'gm_gas_challenge')
plt.show()

## 9. Summary statistics table

In [ ]:
summary_rows = []
for method in methods:
    for param in params:
        for cond in conditions:
            sub = df[(df['method']==method)&(df['parameter']==param)&(df['condition']==cond)]
            if not len(sub): continue
            summary_rows.append({
                'method': method, 'parameter': param, 'condition': cond,
                'n_subjects'  : len(sub),
                'group_mean'  : sub['mean'].mean(),
                'group_sd'    : sub['mean'].std(),
                'group_median': sub['mean'].median(),
            })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(FIG_DIR, 'group_summary.csv'), index=False)
units = dict(zip(CONFIG['param_names'], CONFIG['param_units']))

for method in methods:
    print(f'\n-- {method} --')
    print(f'  {"Param":<6}  ' + '  '.join(f'{c:>20}' for c in conditions))
    for param in params:
        parts = [f'  {param:<6}  ']
        for cond in conditions:
            row = df_summary[(df_summary['method']==method)&
                             (df_summary['parameter']==param)&
                             (df_summary['condition']==cond)]
            if len(row):
                parts.append(f'{row["group_mean"].values[0]:8.2f} +/- {row["group_sd"].values[0]:.2f} {units.get(param,""):>2}')
            else:
                parts.append('        N/A         ')
        print('  '.join(parts))

print(f'\nCSV saved: {FIG_DIR}/group_summary.csv')

In [ ]:
# ── GM quantification table: mean ± SD per method / parameter / condition ─
pivot_rows = []
for method in methods:
    for param in params:
        row = {'Method': method, 'Parameter': get_param_label(param)}
        for cond in conditions_gm:
            sub = df_gm[(df_gm['method']   == method) &
                        (df_gm['parameter'] == param)  &
                        (df_gm['condition'] == cond)]
            if len(sub):
                m = sub['mean'].mean()
                s = sub['mean'].std()
                row[COND_LABELS[cond].replace('\n', ' ')] = f'{m:.2f} ± {s:.2f}'
            else:
                row[COND_LABELS[cond].replace('\n', ' ')] = 'N/A'
        # Append significance stars for each pair
        for (c1, c2), r in STATS_GM[method][param].items():
            row[f'{c1} vs {c2}'] = r.get('stars', 'ns')
        pivot_rows.append(row)

df_table = pd.DataFrame(pivot_rows)
df_table.to_csv(os.path.join(FIG_DIR, 'gm_quantification_table.csv'), index=False)
display(df_table)

## 10. Export significance table

In [ ]:
sig_rows = []
for method in methods:
    for param in params:
        for (c1, c2), r in STATS[method][param].items():
            sig_rows.append({
                'method'   : method, 'parameter': param,
                'cond_a'   : c1,     'cond_b'   : c2,
                'n'        : r.get('n', np.nan),
                'statistic': r.get('statistic', np.nan),
                'p_raw'    : r.get('p_raw', np.nan),
                'p_adj'    : r.get('p_adj', np.nan),
                'stars'    : r.get('stars', 'ns'),
                'test'     : CONFIG['stat_test'],
                'fdr'      : CONFIG['fdr_correct'],
            })

df_sig = pd.DataFrame(sig_rows)
sig_path = os.path.join(FIG_DIR, 'significance_table.csv')
df_sig.to_csv(sig_path, index=False)
print(f'Significance table saved: {sig_path}')
df_sig